In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Installazione dipendenze aggiornate
!pip install lm-eval -q
!pip install peft -q
!pip install sortedcontainers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cu

In [2]:
# === CONFIGURA QUI ===
KAGGLE_DATASET_PATH = "/kaggle/input/datasets/lorenzosalis/finetuned-gemma-2b-it/final" 
BASE_MODEL_ID       = "google/gemma-2-2b-it"  # CAMBIATO: Deve essere Gemma 2 Instruct, la vera base dell'adapter!

# Tasks di valutazione
TASKS = "arc_challenge,hellaswag,winogrande,truthfulqa_mc2,mmlu"

NUM_FEWSHOT = 0      # 0 = zero-shot (ideale per testare l'allineamento DPO)
LIMIT       = 250    # 250 è ottimo per un test rapido e sicuro su Kaggle

# === MAPPE METRICHE ===
METRIC_MAP = {
    "arc_challenge": "acc_norm,none", 
    "hellaswag": "acc_norm,none",
    "winogrande": "acc,none",
    "truthfulqa_mc2": "acc,none",
    "mmlu": "acc,none"
}

# Baseline ufficiale di Gemma 2 2B Instruct (nota: molto più alta rispetto alla versione 1!)
OFFICIAL_BASELINE = {
    "arc_challenge": 0.524, 
    "hellaswag": 0.742,
    "winogrande": 0.715,
    "truthfulqa_mc2": 0.450,
    "mmlu": 0.528
}

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Autenticato su HuggingFace")

✅ Autenticato su HuggingFace


In [4]:
import torch
import gc
import json
from lm_eval import evaluator
from lm_eval.models.huggingface import HFLM

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

free_memory()
print(f"GPU libera prima del caricamento: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# Inizializzazione del modello (CORRETTA: apply_chat_template rimossa da qui)
model_ft = HFLM(
    pretrained=BASE_MODEL_ID,
    peft=KAGGLE_DATASET_PATH,
    dtype="float16",              # FP16 per stabilità e velocità di calcolo su T4/P100
    device="cuda",
    max_length=1024,              
)

print(f"GPU dopo caricamento: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB libera")

# Avvio valutazione (CORRETTA: apply_chat_template inserito correttamente qui)
results_ft = evaluator.simple_evaluate(
    model=model_ft,
    tasks=TASKS.split(","),
    num_fewshot=NUM_FEWSHOT,
    batch_size="auto",            # Rileva automaticamente il batch size ideale
    limit=LIMIT,
    apply_chat_template=True,     # Spostato qui: parametro della funzione di valutazione!
)

# Salvataggio dei risultati in un file JSON
with open("/kaggle/working/results_finetuned.json", "w") as f:
    json.dump(results_ft["results"], f, indent=2)

del model_ft
free_memory()
print(f"GPU dopo cleanup: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB libera")
print("Fine-tuned Benchmark: DONE ✓")

# Stampa dei risultati finali con calcolo del Delta
print(f"\n{'Task':<20} {'Baseline (IT Ufficiale)':>24} {'Fine-tuned DPO':>16} {'Delta':>8}")
print("-" * 72)

ft_data = results_ft["results"]

for task, metric in METRIC_MAP.items():
    baseline_score = OFFICIAL_BASELINE.get(task)
    
    try:
        ft_score = ft_data[task].get(metric) 
        if ft_score is None:
            fallback_metric = metric.split(",")[0]
            ft_score = ft_data[task].get(fallback_metric)
    except KeyError:
        ft_score = None

    if baseline_score and ft_score is not None:
        delta = ft_score - baseline_score
        arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
        print(f"{task:<20} {baseline_score:>24.1%} {ft_score:>16.1%} {arrow}{abs(delta):>6.1%}")
    else:
        print(f"{task:<20} {'N/A':>24} {str(ft_score or 'N/A'):>16}")

GPU libera prima del caricamento: 15.53 GB


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/171 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/19 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Running loglikelihood requests: 100%|██████████| 45179/45179 [25:14<00:00, 29.83it/s] 
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


GPU dopo cleanup: 10.17 GB libera
Fine-tuned Benchmark: DONE ✓

Task                  Baseline (IT Ufficiale)   Fine-tuned DPO    Delta
------------------------------------------------------------------------
arc_challenge                           52.4%            40.8% ▼ 11.6%
hellaswag                               74.2%            45.2% ▼ 29.0%
winogrande                              71.5%            65.6% ▼  5.9%
truthfulqa_mc2                          45.0%            58.1% ▲ 13.1%
mmlu                                    52.8%            30.1% ▼ 22.7%
